In [ ]:
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

with open('/content/drive/MyDrive/Dataset/Sentiment/Res_ABSA/Res_ABSA/Train.txt', 'r') as file:
    # Đọc nội dung của tệp và lưu vào biến content
    content = file.read()
    lines = content.split('\n\n')
    data = [line.split('\n') for line in lines]
    df_train = pd.DataFrame(data, columns=['id', 'text', 'label'])
    df_train = df_train.drop(['id'], axis=1)

with open('/content/drive/MyDrive/Dataset/Sentiment/Res_ABSA/Res_ABSA/Test.txt', 'r') as file:
    # Đọc nội dung của tệp và lưu vào biến content
    content = file.read()
    lines = content.split('\n\n')
    data = [line.split('\n') for line in lines]
    df_test = pd.DataFrame(data, columns=['id', 'text', 'label'])
    df_test = df_test.drop(['id'], axis=1)

with open('/content/drive/MyDrive/Dataset/Sentiment/Res_ABSA/Res_ABSA/Dev.txt', 'r') as file:
    # Đọc nội dung của tệp và lưu vào biến content
    content = file.read()
    lines = content.split('\n\n')
    data = [line.split('\n') for line in lines]
    df_val = pd.DataFrame(data, columns=['id', 'text', 'label'])
    df_val = df_val.drop(['id'], axis=1)

In [ ]:
import re
import torch

# Define the mappings
aspect_to_index = {'ambience#general': 0, 'drinks#prices': 1, 'drinks#quality': 2, 'drinks#style&options': 3, 'food#prices': 4,
                   'food#quality': 5, 'food#style&options': 6, 'location#general': 7, 'restaurant#general': 8, 'restaurant#miscellaneous': 9,
                   'restaurant#prices': 10, 'service#general': 11}
# Adding 'None' to handle unspecified sentiments for any detected aspect
sentiment_to_index = {'positive': 1, 'negative': 0, 'neutral': 0, 'none': 0}

# Preprocess the labels
def preprocess_labels(label_str):
    labels = re.findall(r'\{(.*?)\}', label_str.lower())
    aspects = torch.zeros(len(aspect_to_index), dtype=torch.long)
    sentiments = torch.full((len(aspect_to_index),), sentiment_to_index['none'], dtype=torch.long)

    # Initialize with 'None'
    for label in labels:
        if ', ' in label:
            aspect, sentiment = label.split(', ')
            if aspect in aspect_to_index and sentiment in sentiment_to_index:  # Only process known aspects with sentiments
                idx = aspect_to_index[aspect]
                aspects[idx] = 1
                sentiments[idx] = sentiment_to_index[sentiment]

    return aspects, sentiments

df_train['aspects'], df_train['sentiments'] = zip(*df_train['label'].apply(preprocess_labels))
df_test['aspects'], df_test['sentiments'] = zip(*df_test['label'].apply(preprocess_labels))
df_val['aspects'], df_val['sentiments'] = zip(*df_val['label'].apply(preprocess_labels))

In [ ]:
from bs4 import BeautifulSoup

for df in [df_train, df_test, df_val]:
  for sentence in range(0, len(df.text)):
    # xóa tag, link http
    processed_feature = BeautifulSoup(str(df.text[sentence]), "html.parser").get_text()
    #email-id
    processed_feature = re.sub('b[w-]+?@w+?.w{2,4}b', 'emailadd', processed_feature)
    #url
    processed_feature = re.sub('(http[s]?S+)|(w+.[A-Za-z]{2,4}S*)', 'urladd', processed_feature)

    # Xóa tất cả các ký tự đặc biệt
    processed_feature = re.sub(r'\W', ' ', processed_feature)

    # xóa từ có chứa chữ số
    # processed_feature = re.sub('W*dw*', '', processed_feature)

    # Chuyển đổi sang chữ thường
    processed_feature = processed_feature.lower()

    # Thay thế nhiều khoảng trắng bằng một khoảng trắng
    processed_feature = re.sub(r'\s+', ' ', processed_feature, flags=re.I)

    # processed_features.append(processed_feature)
    df.text[sentence] = processed_feature

In [ ]:
df_train['aspects'] = df_train['aspects'].apply(lambda x: x.numpy())
df_train['sentiments'] = df_train['sentiments'].apply(lambda x: x.numpy())

df_test['aspects'] = df_test['aspects'].apply(lambda x: x.numpy())
df_test['sentiments'] = df_test['sentiments'].apply(lambda x: x.numpy())

df_val['aspects'] = df_val['aspects'].apply(lambda x: x.numpy())
df_val['sentiments'] = df_val['sentiments'].apply(lambda x: x.numpy())

In [ ]:
df_train

,text,label,aspects,sentiments
0,giá 53k size vừa,"{DRINKS#PRICES, neutral}, {DRINKS#STYLE&OPTION...","[0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
1,nhưng nói chung cũng hơi đắt,"{RESTAURANT#PRICES, negative}","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
2,mình ăn rất hôi mùi dầu,"{FOOD#QUALITY, negative}","[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
3,mình ăn chưa baoh thấy mùi hôi hải sản,"{FOOD#QUALITY, positive}","[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]"
4,3 dĩa vs 2 lon revive mà có 190k thui,"{RESTAURANT#PRICES, positive}","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]"
...,...,...,...,...
7023,cơ mà hôm đó bạn nv order có vẻ khó chịu với m...,"{SERVICE#GENERAL, negative}","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
7024,tổng hoá đơn phần bò sốt là 115k,"{FOOD#PRICES, neutral}","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
7025,quán trang trí bắt mắt và sáng sủa,"{AMBIENCE#GENERAL, positive}","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
7026,trà đào ở đây đậm đà hơn ngon hơn mấy chỗ khác...,"{DRINKS#QUALITY, positive}, {RESTAURANT#MISCEL...","[0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0]","[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]"


In [ ]:
import gensim

documents = [_text.split() for _text in df_train.text]
w2v_model = gensim.models.word2vec.Word2Vec(vector_size=128,
                                            window=7,
                                            min_count=10,
                                            workers=8)

w2v_model.build_vocab(documents)

w2v_model.train(documents, total_examples=len(documents), epochs=32)

(2701076, 3666912)

In [ ]:
words = w2v_model.wv
vocab_size = len(words)
print("Vocab size", vocab_size)

Vocab size 1147


In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer()
tokenizer.fit_on_texts(df_train.text)

vocab_size = len(tokenizer.word_index) + 1
print("Total words", vocab_size)

Total words 4491


In [ ]:
embedding_matrix = np.zeros((vocab_size, 128))
for word, i in tokenizer.word_index.items():
  if word in w2v_model.wv:
    embedding_matrix[i] = w2v_model.wv[word]
print(embedding_matrix.shape)

(4491, 128)


In [ ]:
X_train = pad_sequences(tokenizer.texts_to_sequences(df_train.text.values), maxlen=128)
X_test = pad_sequences(tokenizer.texts_to_sequences(df_test.text.values), maxlen=128)
X_val = pad_sequences(tokenizer.texts_to_sequences(df_val.text.values), maxlen=128)

In [ ]:
aspect_train = np.stack(df_train.aspects.values)
aspect_test = np.stack(df_test.aspects.values)
aspect_val = np.stack(df_val.aspects.values)

sentiment_train = np.stack(df_train.sentiments.values)
sentiment_test = np.stack(df_test.sentiments.values)
sentiment_val = np.stack(df_val.sentiments.values)

In [ ]:
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import BatchNormalization, Conv1D, MaxPooling1D, Activation, Flatten, Dropout, Dense, DepthwiseConv1D, GlobalAveragePooling1D, Input, Embedding

import tensorflow.keras.backend as K

INPUT_SHAPE = (X_train.shape[1],)
N_ASPECT = 12
N_SENTIMENT = 12

class AspectSentimentNet(object):
  def _aspect_basenetwork(inputs, classes = N_ASPECT, finAct = 'softmax'):
    embedding = Embedding(input_dim=vocab_size,
                      output_dim=128,
                      weights=[embedding_matrix],
                      trainable=True)(inputs)
    # CONV => RELU => POOL
    x = Conv1D(filters=128, kernel_size=3, padding='same', activation='relu')(embedding)
    x = BatchNormalization(axis=-1)(x)
    x = MaxPooling1D(pool_size=3)(x)
    x = Dropout(0.3)(x)

    # CONV => RELU => POOL
    x = Conv1D(filters=128, kernel_size=3, padding='same', activation='relu')(x)
    x = BatchNormalization(axis=-1)(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    # CONV => RELU => POOL
    x = Conv1D(filters=128, kernel_size=3, padding='same', activation='relu')(x)
    x = BatchNormalization(axis=-1)(x)
    x = MaxPooling1D(pool_size=2)(x)

    # MLP classifier
    x = Flatten()(x)
    x = Dense(128, activation='relu')(x)
    x = BatchNormalization(axis=-1)(x)
    x = Dense(classes)(x)
    x = Activation(finAct, name="aspect_output")(x)

    return x

  def _sentiment_basenetwork(inputs, classes = N_SENTIMENT, finAct = 'softmax'):
    embedding = Embedding(input_dim=vocab_size,
                          output_dim=128,
                          weights=[embedding_matrix],
                          trainable=True)(inputs)
    # CONV => RELU => POOL
    x = Conv1D(filters=128, kernel_size=3, padding='same', activation='relu')(embedding)
    x = BatchNormalization(axis=-1)(x)
    x = MaxPooling1D(pool_size=3)(x)
    x = Dropout(0.3)(x)

    # CONV => RELU => POOL
    x = Conv1D(filters=128, kernel_size=3, padding='same', activation='relu')(x)
    x = BatchNormalization(axis=-1)(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    # CONV => RELU => POOL
    x = Conv1D(filters=128, kernel_size=3, padding='same', activation='relu')(x)
    x = BatchNormalization(axis=-1)(x)
    x = MaxPooling1D(pool_size=2)(x)

    # MLP classifier
    x = Flatten()(x)
    x = Dense(128, activation='relu')(x)
    x = BatchNormalization(axis=-1)(x)
    x = Dense(classes)(x)
    x = Activation(finAct, name="sentiment_output")(x)

    return x

  @staticmethod
  def build(inputShape=INPUT_SHAPE, numAspect = N_ASPECT, numSentiment = N_SENTIMENT):
    inputs = Input(shape=INPUT_SHAPE)
    aspectBranch=AspectSentimentNet._aspect_basenetwork(inputs=inputs,
      classes = numAspect, finAct='softmax')
    sentimentBranch=AspectSentimentNet._sentiment_basenetwork(inputs=inputs,
      classes = numSentiment, finAct='softmax')

    # Tạo một mô hình sử dụng đầu vào là chuỗi embedding, sau đó mô hình sẽ rẽ nhánh, một nhánh xác định đặc trưng của aspect và một nhánh xác định đặc trưng của sentiment
    model = Model(inputs=inputs,
                  outputs=[aspectBranch, sentimentBranch],
                  name="aspect_sentiment_net")

    return model

model = AspectSentimentNet.build(inputShape=INPUT_SHAPE, numAspect = N_ASPECT, numSentiment = N_SENTIMENT)


In [ ]:
from tensorflow.keras.optimizers import AdamW

LR_RATE = 5e-5
EPOCHS = 100
opt = AdamW(learning_rate=LR_RATE, weight_decay=LR_RATE / EPOCHS)

losses = {"aspect_output": "binary_crossentropy",
          "sentiment_output": "categorical_crossentropy"}
metrics = {"aspect_output": "accuracy",
           "sentiment_output": "accuracy"}
model.compile(loss=losses, optimizer=opt, metrics=metrics)


In [ ]:
history = model.fit(X_train, {"aspect_output": aspect_train, "sentiment_output": sentiment_train},
                    validation_data=(X_val, {"aspect_output": aspect_val, "sentiment_output": sentiment_val}),
                    batch_size=32,
                    steps_per_epoch=len(X_train) // 32,
                    epochs=EPOCHS, verbose=1)

Epoch 1/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 48s 172ms/step - aspect_output_accuracy: 0.1014 - aspect_output_loss: 0.8420 - loss: 3.1066 - sentiment_output_accuracy: 0.0896 - sentiment_output_loss: 2.2646 - val_aspect_output_accuracy: 0.0830 - val_aspect_output_loss: 0.6502 - val_loss: 2.4059 - val_sentiment_output_accuracy: 0.2477 - val_sentiment_output_loss: 1.7958
Epoch 2/100
  1/219 ━━━━━━━━━━━━━━━━━━━━ 21s 99ms/step - aspect_output_accuracy: 0.1000 - aspect_output_loss: 0.7677 - loss: 3.2163 - sentiment_output_accuracy: 0.1500 - sentiment_output_loss: 2.4485

/usr/lib/python3.10/contextlib.py:153: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(typ, value, traceback)


219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - aspect_output_accuracy: 0.1000 - aspect_output_loss: 0.3856 - loss: 3.2163 - sentiment_output_accuracy: 0.1500 - sentiment_output_loss: 1.2299 - val_aspect_output_accuracy: 0.0843 - val_aspect_output_loss: 0.6501 - val_loss: 2.4051 - val_sentiment_output_accuracy: 0.2464 - val_sentiment_output_loss: 1.7949
Epoch 3/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 43s 188ms/step - aspect_output_accuracy: 0.1587 - aspect_output_loss: 0.7505 - loss: 2.6011 - sentiment_output_accuracy: 0.1649 - sentiment_output_loss: 1.8506 - val_aspect_output_accuracy: 0.2296 - val_aspect_output_loss: 0.6334 - val_loss: 2.1425 - val_sentiment_output_accuracy: 0.3217 - val_sentiment_output_loss: 1.5508
Epoch 4/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - aspect_output_accuracy: 0.1500 - aspect_output_loss: 0.3476 - loss: 2.6649 - sentiment_output_accuracy: 0.0500 - sentiment_output_loss: 0.9910 - val_aspect_output_accuracy: 0.2322 - val_aspect_output_loss: 0.6333 - val_loss: 2.1

In [ ]:
sentence = "Nhà hàng có không gian đẹp, nhưng chất lượng món ăn lại không như mong đợi, phục vụ hơi chậm."
input = pad_sequences(tokenizer.texts_to_sequences([sentence]), maxlen=128)
prediction = model.predict(input)
print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
[array([[3.9506879e-01, 1.8387154e-04, 1.2747071e-03, 1.6088695e-04,
        5.9021369e-04, 3.5629950e-02, 1.7868053e-03, 6.1998289e-04,
        7.1115769e-03, 1.9859204e-03, 2.5414340e-03, 5.5304581e-01]],
      dtype=float32), array([[0.23757823, 0.03089561, 0.04933604, 0.04820806, 0.04429764,
        0.08641684, 0.08543905, 0.04576539, 0.18610704, 0.07275529,
        0.05376234, 0.05943847]], dtype=float32)]


In [ ]:
score = model.evaluate(X_test, {"aspect_output": aspect_test, "sentiment_output": sentiment_test}, batch_size=32)


61/61 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - aspect_output_accuracy: 0.6649 - aspect_output_loss: 0.1484 - loss: 1.3902 - sentiment_output_accuracy: 0.3874 - sentiment_output_loss: 1.2419



In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_pred = model.predict(X_test)

aspect_pred_binary = (y_pred[0] > 0.5).astype(int)

precision_aspect = precision_score(aspect_test, aspect_pred_binary, average='micro')
recall_aspect = recall_score(aspect_test, aspect_pred_binary, average='micro')
f1_aspect = f1_score(aspect_test, aspect_pred_binary, average='micro')

sentiment_pred_class = (y_pred[0] > 0.5).astype(int)
precision_sentiment = precision_score(sentiment_test,sentiment_pred_class, average='micro')
recall_sentiment = recall_score(sentiment_test,sentiment_pred_class, average='micro')
f1_sentiment = f1_score(sentiment_test,sentiment_pred_class, average='micro')

print("Aspect Precision, Recall, F1:", precision_aspect, recall_aspect, f1_aspect)
print("Sentiment Precision, Recall, F1:", precision_sentiment, recall_sentiment, f1_sentiment)

61/61 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step
Aspect Precision, Recall, F1: 0.8524096385542169 0.5382274629136554 0.6598274656096992
Sentiment Precision, Recall, F1: 0.5240963855421686 0.5788423153692615 0.5501106544419855


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluation(aspect_test, sentiment_test):
  y_pred = model.predict(X_test)

  aspect_pred_binary = (y_pred[0] > 0.5).astype(int)

  precision_aspect = precision_score(aspect_test, aspect_pred_binary, average='micro')
  recall_aspect = recall_score(aspect_test, aspect_pred_binary, average='micro')
  f1_aspect = f1_score(aspect_test, aspect_pred_binary, average='micro')

  sentiment_pred_class = (y_pred[0] > 0.5).astype(int)
  precision_sentiment = precision_score(sentiment_test,sentiment_pred_class, average='micro')
  recall_sentiment = recall_score(sentiment_test,sentiment_pred_class, average='micro')
  f1_sentiment = f1_score(sentiment_test,sentiment_pred_class, average='micro')

  precision = (precision_aspect + precision_sentiment) / 2
  recall = (recall_aspect + recall_sentiment) / 2
  f1 = (f1_aspect + f1_sentiment) / 2

  print("Aspect, Sentiment and Total respectively")
  print("Precision:", precision_aspect, precision_sentiment, precision)
  print("Recall:", recall_aspect, recall_sentiment, recall)
  print("F1:", f1_aspect, f1_sentiment, f1)

evaluation(aspect_test, sentiment_test)

61/61 ━━━━━━━━━━━━━━━━━━━━ 3s 45ms/step
Aspect, Sentiment and Total respectively
Precision: 0.8524096385542169 0.5240963855421686 0.6882530120481928
Recall: 0.5382274629136554 0.5788423153692615 0.5585348891414584
F1: 0.6598274656096992 0.5501106544419855 0.6049690600258424
